# EGB103 Assignment One - Computational Fluid Dynamics (Virtual Wind Tunnel)

Programming and software play a crucial role in helping engineers solve problems such as designing and optimizing aerodynamic vehicles. These tools allow engineers to perform complex calculations, simulate dynamic behaviours, and visualise results efficiently. Through programming, engineers can automate repetitive tasks, implement numerical methods, and find optimal solutions to design challenges.

<img src="images/header.png" width=600>

In this assignment, you will learn Python in the context of an authentic engineering problem by creating a simple virtual wind tunnel.


<div style="background-color: #00ccff; padding:10px">

## Part A - Due Friday 13th March (end of week 3)

- Task 1 - *Add markdown*
- Task 2 - *Implement function relax_towards*

## Part B - Due Moday 13th April (start of week 7)

- Task 3 - *Implement Helper functions*
- Task 4 - *Implement Simulation functions*
- Task 5 - *Experiment with Model Objects in the Wind Tunnel*
</div>

<p></p>

<div style="background-color: #ffcc00; padding:10px">

# General Rules and Restrictions

- You must develop your solution entirely with Jupyter in the cloud on https://jupyter.eres.qut.edu.au
- For the assignments you cannot work with friends or colleagues, or get help from anyone other than the EGB103 teaching team - it needs to be entirely your own work.
- You cannot use AI tools such as ChatGPT or Copilot to help develop your solution.
- Do not modify the name or list of parameters for any of the functions.
- Every function must be implemented in only one place within this .ipynb file (where it is already defined). Do not cut and paste to create duplicate definitions of the same function.
- Do not add any import statements. The only modules you are allowed to import are the ones I have imported already in the skeleton solution, namely the math and wind_tunnel modules.
- You should only use Python language features that we have covered in lectures or tutorials. If you use other features we will suspect acadamic misconduct and require you to attend an oral viva to authenticate your learning.
- Do not create any additional functions, just implement the functions already defined.
- Do not create or make use of any global variables in your functions. If you need to define a constant, make it local to your function.
</div>

<p></p>

<div style="background-color: #00cc00; padding:10px">

# Part A

## Task 1: Add markdown to list your name and previous programming experience

</div>    
<p></p>

Create a markdown cell ***immediately below this cell*** that includes your name and briefly describes your previous programming experience if any.

Must include some markdown to style or format in some manner.


<div style="background-color: #00cc00; padding:10px">

## Task 2: Implement function relax_towards
</div>    

<p></p>
The <code>relax_towards</code> function below implements the relaxation formula to compute the next value by relaxing the current value towards the equilibrium value:

$$ f_{\text{next}} = f_{\text{current}} - \frac{1}{\tau}\big(f_{\text{current}} - f_{\text{eq}}\big)$$

In [ ]:
# Function Overview: Applies relaxation towards an equilibrium value, as used in LBM collide step.
#
# Makes use of: None
#
# Parameters:
#    current (float): The current distribution value for a given direction in a cell.
#    equilibrium (float): The equilibrium distribution value for the same direction.
#    relaxation_time (float): The relaxation time parameter (τ) controlling the rate of convergence.
#
# Returns:
#    float: The relaxed distribution value after applying the LBM collision formula.
#
def relax_towards(current, equilibrium, relaxation_time):
    pass # delete this line and replace it by your actual implementation      

In [ ]:
# Task 2: Test function relax_towards

print(f'relax_towards(0.4, 0.45, 0.6) returns:', relax_towards(0.4, 0.45, 0.6), '(expected 0.48333333333333334)')      
# Notice in this case we "overshoot" the equilibrium, but the gap between the current value and the equilibrium has reduced from (0.45 - 0.40) = 0.05 to 0.033333333333333326, i.e. we are getting closer to equilibrium.
# This overshoot happens when tau < 1

print(f'relax_towards(0.4, 0.45, 1.5) returns:', relax_towards(0.4, 0.45, 1.5), '(expected 0.43333333333333335)')      


# Perform additional extensive tests on function relax_towards
import test_correctness
ok = test_correctness.run_tests(relax_towards)

<div style="background-color: #00cc00; padding:10px">

# Part B - About Computational Fluid Dynamics (CFD)

</div>

***Please read this section before attempting to implement any of the functions in Part B***.

In this assignment, you will learn Python in the context of an authentic engineering problem, by creating a simple virtual wind tunnel. 
To keep things manageable, our simulation will work only in 2D. We will use Computational Fluid Dynamics (CFD) techniques to analyse airflow. CFD can model the flow of fluids such as water ($\text{H}_2\text{O}$) and gases such as atmospheric air.

Most CFD methods start by dividing the space being modelled into a finite number of regular-sized *cells*. In 2D these cells are squares; in 3D they are cubes. For a given spatial domain, dividing it into more, smaller cells produces results closer to physical reality, but increases computational complexity - your program will take longer to run. Below is an illustration of our virtual 2D wind tunnel subdivided into cells. The fine red lines show cell boundaries. Some cells represent solid, immovable obstacles that block airflow. These obstacles are shown in black and include the floor and ceiling of the tunnel, the curved manifolds at each end that help air circulate, and the model being tested (in this example, a very simple F1 car).

<img src="images/grid.png" width=700>

CFD simulations progress through a sequence of time steps. The system starts in an initial state, and we compute how that state changes after one time step. Time steps must be small so that only minor changes occur between steps. Smaller time steps improve accuracy but make the program slower.

## The Lattice Boltzmann Method

The “*gold standard*” for fluid dynamics is <a href="https://en.wikipedia.org/wiki/Navier%E2%80%93Stokes_equations">Navier–Stokes</a>, but we will use the ***Lattice Boltzmann Method (LBM)***, which is simpler yet widely used for its computational efficiency. LBM maintains a ***distribution*** of air molecules across the spatial domain. Some cells contain more molecules than others, giving them higher ***density*** or *pressure*. This distribution changes over time as air molecules move.

LBM classifies molecules within each cell by their direction of movement. Even when air seems still, individual molecules are always moving. If equal numbers move in all directions, the cell’s net ***velocity*** will be zero, though individual molecules still move. We ***classify*** molecules based on ***where*** they will be after the next ***time step***:

* Molecules that will end up in the cell one to the right are classified as moving **East**.
* Molecules that will end up in the cell diagonally up one and to the right are classified as moving **NorthEast**.
* And so on for **North**, **South**, **West**, **NorthWest**, **SouthEast**, **SouthWest**.  
* Molecules that remain in the same cell are classified as at **Rest**.

Each cell stores ***nine*** values (<a href="https://en.wikipedia.org/wiki/Lattice_Boltzmann_methods#Lattices_and_the_DnQm_classification">D2Q9</a>): the ***mass*** of air moving in each 8 directions plus those at rest:

<img src="images/cell.png" width=180>

The cell’s total ***density*** is the sum of these nine values. For example, the cell above has a density of $0.990$.

To compute a cell's ***velocity***, we calculate the horizontal (x) and vertical (y) components separately:

* For the ***x*** component, we sum the ***Easterly*** components (**East**, **NorthEast**, **SouthEast**) and subtract the ***Westerly*** components (**West**, **NorthWest**, **SouthWest**).
* For the ***y*** component, we sum the ***Northerly*** components (**North**, **NorthEast**, **NorthWest**) and subtract the ***Southerly*** components (**South**, **SouthEast**, **SouthWest**).  
* The **Rest** component does not affect velocity.

We then need to divide those quantities by the density of the cell. For the above example:

> $v_x = (0.020 + 0.125 + 0.029 - 0.017 - 0.105 - 0.028)/0.990 \approx 0.02424$  
> $v_y = (0.017 + 0.095 + 0.020 - 0.028 - 0.121 - 0.029)/0.990 \approx -0.04646$

So, the resulting cell velocity is in a *South Easterly* direction with a ***speed*** of $\approx 0.0524$. 

## Streaming

As we know, each cell has a distribution of air molecules moving in each of the $8$ directions. At each ***time step***, those molecules move or ***stream*** from one cell to the *next* based on the direction they are currently heading. 
So, all the molecules currently heading **East** move to the cell to the right and become the new value for the **East** streaming direction of that cell. 
Meanwhile the molecules which were heading **East** in that cell to our right have moved on/streamed to the cell to their right. 
The molecules heading **NorthEast** move to the cell diagonally up and to the right and become the new value for the **NorthEast** distribution of that cell, and so on for each of the other directions. The molecules that are at **Rest**, stay put in the current cell and remain at **Rest**. 

In the diagram below we have an example $5 \times 5$ grid of cells. The cells coloured black are solid immovable obstacles that our air molecules cannot move to or through. The grid on the left depicts a hypothetical initial distribution at time $t$. 
The grid to the right depicts where those corresponding distribution values will have ***moved*** to at time step $t+1$ as a result of this ***streaming*** process. 
Each cell has $9$ ***numbers*** associated with it, the molecules at **Rest** (shown in the centre of each cell) and those moving in each of the $8$ directions. 
Instead of showing actual *numbers* I have replaced each number by a *label*. 
For example **b1** presents the number of molecules streaming **East** in cell **a** at time step $t$. 
As we can see, at time step $t+1$, this value has *moved* to become the **East** streaming component for cell **c** (in the middle of the grid). 
All of the cell values colour coded in <span style="color:blue">blue</span> correspond to values computed using this streaming process.

<img src="images/stream_or_bounce.png" width=800>

## Bounce Back

However, as we see from this illustration, not all values are able to stream to their corresponding neighbouring cell because there is an ***obstacle*** in the cell where they would have streamed to. 
***Instead*** of streaming into that obstacle cell, they ***bounce*** off that obstacle and begin moving in the ***opposite*** direction. 
Take for example **e1** which is the **East** streaming value of cell **e** at time step $t$. 
It can't stream to the cell immediately to its right because that cell is off the edge of our  grid, which corresponds to an implicit solid obstacle. So, that flow ***bounces*** off that ***wall*** on the right and becomes the **West** flowing component of cell **e** at time $t+1$. 

You might think that assigning a new value to the **West** flowing component of cell **e** might ***conflict*** with the air molecules that would otherwise stream into that cell. 
But if we consider where those molecules would have streamed from, we see they would have streamed from the cell immediately to the right, but this cell is off the edge of our grid (an implicit obstacle cell), so no air molecules would have streamed to us from that cell. 
So, conveniently, the new value of every direction of every cell is determined by either ***streaming*** values from the corresponding neighbouring cells (those cells colour coded <span style="color:blue">blue</span> in our diagram), or they get their value from a flow ***bouncing*** back off a solid obstacle to now be flowing in the ***opposite*** direction (shown in <span style="color:red">red</span> in our diagram). 
We therefore call this process ***stream_or_bounce***. 


## Double Buffering

Notice in the streaming diagram above we had two copies on the entire grid, the one on the left corresponding to what the distribution looked like at time step $t$ and the one on the right corresponding to what it will look like at time step $t+1$. 
Note that both grids (before and after) have the same size and shape ($5 \times 5$ cells) and both have obstacles at the exact same places (obstacles don't move). 
In our program we will have two corresponding Python data structures (lists), one to store the ***current*** state of the distribution and one to store the ***next*** state of the distribution (after streaming). We use two separate grid data structures, because when we are computing the new value of a direction in a cell (whether it be via streaming or bounce back), we always need to use the value that was in that cell at time step $t$. If we tried updating these values *"in place"* by just using one grid data structure that we were both reading and writing, then we might accidentally read a grid value that we just updated (corresponding to the value of that cell distribution at time step $t+1$ rather than the value as it was at time step $t$. We call this process double buffering.

Once we have completed the ***stream_or_bounce*** for the entire grid, we need to ***swap*** our buffers around.
What was the **next buffer** that we just updated to store the distribution at time step $t+1$, becomes the **current buffer**, and what was the **current buffer** (that we just read from), becomes the **next buffer** that we will write over in the next iteration of our simulation loop.

## Equilibrium

***Velocity*** and ***density*** are ***macroscopic*** properties. Many distributions can produce the same velocity and density, but there is one special distribution called *equilibrium* that the system tends to return to. Smooth (*laminar*) flow is close to equilibrium; turbulent flow is far from it. The equilibrium distribution depends only on a cell’s velocity and density.

If the air is still (velocity = $0,0$) and the density is 1 (i.e. standard atmospheric pressure), then the equilibrium values are:  
> $ w_{\text{Rest}} = \frac{4}{9}$  
> $w_{\text{North}} = w_{\text{East}} = w_{\text{South}} = w_{\text{West}} = \frac{1}{9}$  
> $w_{\text{NorthEast}} = w_{\text{NorthWest}} = w_{\text{SouthEast}} = w_{\text{SouthWest}} = \frac{1}{36}$

In general (for any cell velocity and density), the equilibrium value for direction $d$ is:
$$f_d^{\text{eq}} = w_d \, \rho \, \Big( 1 + \lambda_d + \frac{1}{2}\lambda_d^2 - \frac{3}{2}(v_x^2 + v_y^2) \Big)$$
Where:  
> $\lambda_d = 3({\small \Delta}_{x,d}\,v_x + {\small \Delta}_{y,d}\,v_y)$  
> $w_d$ = equilibrium weight for direction $d$ (*when velocity = $0,0$ and density = $1$*)  
> $\rho$ = cell density  
> $v_x, v_y$ = cell velocity (*x and y components*)  
> ${\small \Delta}_{x,d}$ = number of cells we move horizontally when streaming in direction $d$ (either -1, 0 or +1)  
> ${\small \Delta}_{y,d}$ = number of cells we move vertically when streaming in direction $d$ (either -1, 0 or +1) 

## Collide (*Relaxing Towards Equilibrium*)

When new air streams into a cell (from all directions), those streams of air ***collide*** with one another, and these collisions cause the distribution of air molecules moving in each direction within the cell to change.

If we *stir* or *whisk* a liquid, its flow can become turbulent or exhibit large velocity gradients. If the stirring stops, the flow gradually returns to a smoother, more laminar state. This tendency to return to ***equilibrium*** is governed by the fluid’s **viscosity**:  
- A highly viscous fluid retains *eddies* and *swirling* patterns for longer.  
- A less viscous fluid relaxes to equilibrium more quickly.

In the lattice Boltzmann method, this ***relaxation*** is modeled using a *relaxation parameter* ($\tau$), which controls how rapidly each cell returns to its equilibrium distribution. At each ***time step*** we update each cell, for each direction $d$ as follows:

$$ f_d^{\text{next}} = f_d^{\text{current}} - \frac{1}{\tau}\big(f_d^{\text{current}} - f_d^{\text{eq}}\big)$$

For air, we typically use $\tau$ values between $0.5$ and $0.6$. 
Air viscosity is not constant under all flow conditions, at higher speeds (higher Mach numbers), the *effective* viscosity decreases slightly because the flow becomes more *laminar* and less influenced by molecular collisions. In LBM, we mimic this by adjusting $\tau$:

- High-speed flows → lower effective viscosity → $\tau$ closer to $0.5$.
- Low-speed flows → higher effective viscosity → $\tau$ slightly larger (around $0.6$).

The following example illustrates the relaxation process from end to end for a single cell. 
<table style="margin-left:0;margin-right:auto">
<tr>
<th>Before Relaxation:<br><img src="images/cell.png" width=150></th>
<td style="font-size:32px">→</td>
<td>$\rho=0.990$<br>$v\approx(0.0242,-0.0464)$</td>
<td style="font-size:32px">→</td>
<th>Equilibrium:<br><img src="images/feq.png" width=150></th>
<td style="font-size:32px">→</td>
<td>Relax towards equilibrium<br>$(\tau=0.55$)</td>
<td style="font-size:32px">→</td>
<th>After Relaxation:<br><img src="images/relaxed.png" width=150></th>
<td>$\rho=0.990$<br>$v\approx(0.0242,-0.0464)$</td>   
</tr>
</table>

Notice how the cell's density and velocity do not change, only the distribution within the cell.


## Injecting Fan Force to Generate Wind

Our wind tunnel needs some wind! The wind is going to be generated by a virtual fan (shown in yellow in the diagram below).
It will suck air from its left and blow it to the right.

<img src="images/fan.png" width=400>

The following process is applied only to the narrow column of cells where the fan is located. 
For each fan cell we start by calculating the current cell velocity.
If the *x* component of the calculated velocity is less than the fan speed, then we ***boost*** it up to the fan speed.
We then use that boosted velocity when calculating the cell's equilibrium distribution and then relax the cell towards that equilibrium in the normal manner. In this way, we drive the horizontal velocity of those cells towards the speed of the fan.

The following example illustrates the slightly revised ***collide*** process for an example fan cell (assuming a fan speed of $0.05$): 
<table style="margin-left:0">
<tr>
    <th>Before:</th>
    <td></td>
    <th>Current velocity:</th>
    <td></td>
    <th>Apply Fan Force</th>
    <td></td>
    <th>Equilibrium:</th>
    <td></td>
    <td></td>
    <td></td>
    <th>After:</th>
</tr>
<tr>
<td><img src="images/cell.png"></td>
<td style="font-size:32px">→</td>
<td>$\rho=0.990$<br>$v\approx(0.0242,-0.0464)$</td>
<td style="font-size:32px">→</td>
<td>$\rho=0.990$<br>$v\approx(0.05,-0.0464)$</td>    
<td style="font-size:32px">→</td>    
<td><img src="images/fan_cell.png"></td>
<td style="font-size:32px">→</td>
<td>Relax<br>$(\tau=0.55)$</td>    
<td style="font-size:32px">→</td>
<td><img src="images/final_fan.png"></td>  
</tr>
</table>

We apply this fan force only to the cells located directly where the fan is located. However, when the air in those fan cells begins moving to the right (and sucking air from the left), that is then going to cause the neighbouring cells to gradually increase their velocity via streaming. 
Intuitively, the air from the fan pushes the air in front of it, so as to create an air wave that eventually circulates around the entire wind tunnel:

<img src="images/circulating.png" width=600>

Notice how the curved manifolds at each end aid in bending the air around these bends.


## Simulating One Time Step

In ***each time step*** of our simulation algorithm we:
1) We perform the ***Stream or Bounce*** step on the entire grid (function **stream_or_bounce_all_cells**). In performing *Stream or Bounce* we *read* from the **current distribution** and we *write* new values to the **next distribution**.

2) We then swap those buffers around: what was the **next distribution** that we just updated to store the distribution at time step $t+1$, becomes the **current distribution** and what was the **current distribution** (that we just read from), becomes the **next distribution** that we will write over in the next iteration of our simulation loop.
3) ***Collide*** the molecules moving in different directions ***within*** each cell, so that the cell's distribution gets *remixed* to be closer to the equilibrium for that cell (function **apply_fan_and_collide_all_cells**). 

## Representing LBM Distributions in Python

A ***distribution*** is a 2D grid of cells. In Python we will represent this as a **list** of *cell* columns, each of which will be a **list** of *cells*. 
So, we have a **list** of **list**s of *cells*.

If we have a variable named <code>distribution</code> that contains such a **list** of **list**s of cells, and we want to access a specific cell at position $(x,y)$, then we would use the Python expression: <code>distribution[x][y]</code>. Distribution is a list of columns so <code>distribution[x]</code> gives us column number $x$. But since each column is a **list** of cells, we can then index into that list of cells to get the cell in row $y$ of column $x$.

However, with LBM, each cell consists of $9$ separate numbers corresponding to each of the D2Q9 directions. 
We will therefore represent each LBM cell  as a Python **list** containing $9$ **float**ing point numbers.
So, the entire distribution becomes then a **list** of **list**s of **list**s of **float**ing point numbers. 
To access the value of the distribution for the cell at position $(x,y)$ in direction $d$, we would use the Python expression: <code>distribution[x][y][d]</code>.

Within each cell, we need to remember which elements in the list correspond to each of our directions. We therefore import the following Enumerated type from the wind_tunnel module:
```python
class Direction(enum.IntEnum):
    REST = 0
    EAST = 1
    NORTH = 2
    WEST = 3
    SOUTH = 4
    NORTHEAST = 5
    NORTHWEST = 6
    SOUTHWEST = 7
    SOUTHEAST = 8
```
This tells us firstly, that element $0$ of the list will always correspond to the particles at **Rest** and element $7$ will always correspond to the particles moving **SouthWest**. However, when accessing such elements, we want to use meaningful names, so rather than accessing: <code>distribution[x][y][3]</code> we should use <code>distribution[x][y][Direction.WEST]</code>. Here <code>Direction.WEST</code> has the value $3$, but it is better to use <code>Direction.WEST</code> rather than just $3$ as it makes clear to anyone reading our code that we are accessing the **West** component of the cell. 
Using a number like $3$ in an expression is referred to as using a ***magic number*** and is considered poor programming practice, as it makes the reader wonder *"what does the value $3$ means in this context?"* The other nice thing about using this Enum type is that we can also use it to create a <code>for</code> loop that iterates over all the different directions.

So, instead of writing a loop using the magic number $9$:
```python
for direction in range(9):
    ...
```
we can instead write:
```python
for direction in Direction:
    ...
```
which will then iterate over however many directions there happens to be in our lists. 
So, if at some future stage we decide to switch from D2Q9 to D2Q5 or D3Q15, then we won't need to change much of our code.

Creating these **list**s of **list**s of **list**s can be a bit overwhelming if you haven't done that kind of thing before, so we provide a function that performs this task for you:
```python
def initialize_distribution(width, height, obstacle_function, pressure=1.0):
    distribution = []
    for x in range(width):
        column = []
        for y in range(height):
            if obstacle_function(x, y):
                column.append(None)
            else:
                cell = []
                for direction in Direction:
                    cell.append(zero_velocity_equilibrium(direction) * pressure)
                column.append(cell)
        distribution.append(column)
    return distribution
```

As you can see, the inner most loop constructs a cell by appending $9$ floating point numbers corresponding to the equilibrium values when the air is still. The $y$ loop then appends these cells to form a column of cells, or None if there is an obstacle in a cell. Finally, the $x$ loop then appends these columns to form the overall 2D distribution.

If you recall, we need two buffers, a current buffer and a next buffer. At the start of the simulation we call the <code>initialize_distribution</code> function to create our current buffer. 
To construct the other buffer (with the same shape and obstacles in the same places), rather than manually constructing the other buffer, we simply create a cloned copy of the current buffer:

```python
    import copy
    
    distribution_current = initialize_distribution(width, height, obstacle_function)
    distribution_next = copy.deepcopy(distribution_current)
```

Once we have created these two buffers at the start of the simulation, we never again need to worry about creating distributions (by appending to lists). We just use expressions such as in the following example to read or update the values of individual elements:

```python

distribution_next[1][2][Direction.NORTH] = ...

... = distribution_current[4][3][Direction.REST]
```
The only function in which you will need to create your own list is within function <code>compute_equilibrium</code> where you will need to create and return a list of numbers.


<div style="background-color: #00cc00; padding:10px">

## Task 3: Implement Helper Functions

</div> 

<p></p>

The following code cell contains definitions for functions zero_velocity_equilibrium, delta_x, delta_y, opposite_direction, compute_density, compute_velocity, compute_speed, compute_equilibrium, compute_width_and_height and get_upstream_cell.

- Do not modify the name or list of parameters or header comments for any of the functions.
- Each of those functions must be implemented in the code cell below and not be copied anywhere else within this .ipynb file. Do not cut and paste to create duplicate definitions of the same function.
- Do not create any additional functions, just implement the functions already defined.
- Do not create or make use of any global variables in your functions. If you need to define a constant, make it local to your function.


In [ ]:
import math
from wind_tunnel import Direction


# Function Overview: Returns the zero-velocity equilibrium weight for a given lattice direction.
#
# Makes use of: None
#
# Parameters:
#    direction (Direction): The lattice direction.
#
# Returns:
#    float: The equilibrium weight corresponding to the specified direction.
#
def zero_velocity_equilibrium(direction):
    pass # delete this line and replace it by your actual implementation    


# Function Overview: Returns the x-component (Δx) of the lattice velocity for a given direction
#                    For example, delta_x(EAST) is +1, while delta_x(SOUTHWEST) is -1
#
# Makes use of: None
#
# Parameters:
#    direction (Direction): The lattice direction. 
#
# Returns:
#    int: The x-component of the discrete velocity, either -1, 0 or +1
#
def delta_x(direction):
    pass # delete this line and replace it by your actual implementation    


# Function Overview: Returns the y-component (Δy) of the lattice velocity for a given direction
#                    For example, delta_y(NORTH) is +1, while delta_y(SOUTHWEST) is -1
#
# Makes use of: None
#
# Parameters:
#    direction (Direction): The lattice direction. 
#
# Returns:
#    int: The y-component of the discrete velocity, either -1, 0 or +1
#
def delta_y(direction):
    pass # delete this line and replace it by your actual implementation    


# Function Overview: Returns the opposite lattice direction for the given direction.
#                    For example the opposite of direction NORTHEAST is direction SOUTHWEST.
#
# Makes use of: None
#
# Parameters:
#    direction (Direction): A lattice direction.
#
# Returns:
#    Direction: The direction opposite to the input
#
def opposite_direction(direction):
    pass # delete this line and replace it by your actual implementation    


# Function Overview: Computes the macroscopic density (ρ) of a cell by summing all distribution components.
#
# Makes use of: None
#
# Parameters:
#    cell (list[float]): The list of distribution values for all directions in a single lattice cell.
#
# Returns:
#    float: The total density of the cell
#
def compute_density(cell):
    pass # delete this line and replace it by your actual implementation    


# Function Overview: Computes the macroscopic velocity vector for a lattice cell in LBM.
#
# Makes use of: compute_density (and optionally makes use of delta_x and delta_y) 
#
# Parameters:
#    cell (list[float]): The list of distribution values for all directions in a single lattice cell.
#
# Returns:
#    tuple[float, float]: The velocity specified via x and y components
#
def compute_velocity(cell):
    pass # delete this line and replace it by your actual implementation    

    
# Function Overview: Computes the macroscopic speed of a lattice cell in LBM.
#
# Makes use of: compute_velocity
#
# Parameters:
#    cell (list[float]): The list of distribution values for all directions in a single lattice cell.
#
# Returns:
#    float: The magnitude of the velocity vector
#
def compute_speed(cell):
    pass # delete this line and replace it by your actual implementation    


# Function Overview: Computes the equilibrium distribution values (f_eq) for a lattice cell using the LBM formula.
#
# Makes use of: delta_x, delta_y, zero_velocity_equilibrium
#
# Parameters:
#    velocity (tuple[float, float]): The macroscopic velocity of the cell.
#    density (float): The macroscopic density (ρ) of the cell.
#
# Returns:
#    list[float]: The equilibrium distribution values for all lattice directions
#
def compute_equilibrium(velocity, density):
    pass # delete this line and replace it by your actual implementation    


# Function Overview: Returns the number of columns (x-dimension) and the number of rows (y-dimension) in the lattice distribution.
#
# Makes use of: None
#
# Parameters:
#    distribution (list of lists): The 2D grid of cells
#
# Returns:
#    tuple[int, int]: The width and height of the lattice, equal to the number of columns and rows in the distribution.
#
def compute_width_and_height(distribution):
    # This function has been provided by the EGB103 teaching team. Do not modify it in any way.
    return len(distribution), len(distribution[0])


# Function Overview: Retrieves the upstream cell for a given position and lattice directions.
#
# Makes use of: delta_x, delta_y, compute_width_and_height
#
# Parameters:
#    distribution (list[list[Optional[list[float]]]]): The lattice structure where
#        distribution[x][y] is either None (obstacle) or a list of per-direction distributions for cell (x, y).
#    x (int): The x-coordinate of the current cell.
#    y (int): The y-coordinate of the current cell.
#    direction (Direction): The lattice direction used to compute the upstream offset.
#
# Returns:
#    Optional[list[float]]: The upstream cell’s per-direction distribution list if the coordinates are in bounds;
#        None if the upstream coordinates fall outside the lattice. 
#        If the upstream cell is an obstacle, the stored value (None) is returned as-is.
#
def get_upstream_cell(distribution, x, y, direction):
    pass # delete this line and replace it by your actual implementation    

print('✅ All Task 3 functions defined successfully (i.e. no syntax errors). Now to see if they work correctly ...')

In [ ]:
# Task 3a: Test function zero_velocity_equilibrium

print('zero_velocity_equilibrium(Direction.REST) returns', zero_velocity_equilibrium(Direction.REST), '(expected 0.4444444444444444)')

print('zero_velocity_equilibrium(Direction.EAST) returns', zero_velocity_equilibrium(Direction.EAST), '(expected 0.1111111111111111)')

print('zero_velocity_equilibrium(Direction.SOUTHEAST) returns', zero_velocity_equilibrium(Direction.SOUTHEAST), '(expected 0.027777777777777776)')

# Perform additional extensive tests on function zero_velocity_equilibrium
import test_correctness
ok = test_correctness.run_tests(zero_velocity_equilibrium, Direction)

In [ ]:
# Task 3b: Test function delta_x

print('delta_x(Direction.REST) returns', delta_x(Direction.REST), '(expected 0)')
      
print('delta_x(Direction.EAST) returns', delta_x(Direction.EAST), '(expected 1)')

print('delta_x(Direction.SOUTHWEST) returns', delta_x(Direction.SOUTHWEST), '(expected -1)')

# Perform additional extensive tests on function delta_x
import test_correctness
ok = test_correctness.run_tests(delta_x, Direction)

In [ ]:
# Task 3c: Test function delta_y

print('delta_y(Direction.REST) returns', delta_y(Direction.REST), '(expected 0)')

print('delta_y(Direction.NORTH) returns', delta_y(Direction.NORTH), '(expected 1)')

print('delta_y(Direction.SOUTHWEST) returns', delta_y(Direction.SOUTHWEST), '(expected -1)')

# Perform additional extensive tests on function delta_y
import test_correctness
ok = test_correctness.run_tests(delta_x, Direction)

In [ ]:
# Task 3d: Test function opposite_direction

print('opposite_direction(Direction.EAST) returns', opposite_direction(Direction.EAST), '(expected WEST)')

print('opposite_direction(Direction.SOUTHEAST) returns', opposite_direction(Direction.SOUTHEAST), '(expected NORTHWEST)')

# Perform additional extensive tests on function opposite_direction
import test_correctness
ok = test_correctness.run_tests(opposite_direction, Direction)

In [ ]:
# Task 3e: Test function compute_density

cell1 = [0.4444444444444444, 0.1111111111111111, 0.1111111111111111, 0.1111111111111111, 0.1111111111111111, 0.027777777777777776, 0.027777777777777776, 0.027777777777777776, 0.027777777777777776]
print(f'compute_density({cell1}) returns', compute_density(cell1), '(expected 1.0)')

cell2 = [0.4, 0.12, 0.13, 0.14, 0.15, 0.026, 0.0277, 0.0278, 0.029]
print(f'compute_density({cell2}) returns', compute_density(cell2), '(expected 1.0505)')

# Perform additional extensive tests on function density
import test_correctness
ok = test_correctness.run_tests(compute_density)

In [ ]:
# Task 3f: Test function compute_velocity

cell1 = [0.4444444444444444, 0.1111111111111111, 0.1111111111111111, 0.1111111111111111, 0.1111111111111111, 0.027777777777777776, 0.027777777777777776, 0.027777777777777776, 0.027777777777777776]
print(f'compute_velocity({cell1}) returns', compute_velocity(cell1), ', expected (0.0, 0.0)')

cell2 = [0.4, 0.12, 0.13, 0.14, 0.15, 0.026, 0.0277, 0.0278, 0.029]
print(f'compute_velocity({cell2}) returns', compute_velocity(cell2), ', expected (-0.019514516896715864, -0.02198952879581151)')

# Perform additional extensive tests on function compute_velocity
import test_correctness
ok = test_correctness.run_tests(compute_velocity, relies_on=[delta_x, delta_y, compute_density])

In [ ]:
# Task 3g: Test function compute_speed

cell1 = [0.4444444444444444, 0.1111111111111111, 0.1111111111111111, 0.1111111111111111, 0.1111111111111111, 0.027777777777777776, 0.027777777777777776, 0.027777777777777776, 0.027777777777777776]
print(f'speed({cell1}) returns', compute_speed(cell1), '(expected 0.0)')

cell2 = [0.4, 0.12, 0.13, 0.14, 0.15, 0.026, 0.0277, 0.0278, 0.029]
print(f'compute_speed({cell2}) returns', compute_speed(cell2), '(expected 0.029399927659333323)')

# Perform additional extensive tests on function compute_speed
import test_correctness
ok = test_correctness.run_tests(compute_speed, relies_on=[compute_velocity, delta_x, delta_y, compute_density])

In [ ]:
# Task 3h: Test function compute_equilibrium

print(f'compute_equilibrium((0, 0), 1) returns:\n', compute_equilibrium((0,0), 1))
print('expected:\n [0.4444444444444444, 0.1111111111111111, 0.1111111111111111, 0.1111111111111111, 0.1111111111111111, 0.027777777777777776, 0.027777777777777776, 0.027777777777777776, 0.027777777777777776]')
print()

print(f'compute_equilibrium((0.02, -0.001), 0.99) returns:\n', compute_equilibrium((0.02,-0.001), 0.99))
print('expected:\n [0.4397353399999999, 0.116731835, 0.10960432999999997, 0.10353183499999997, 0.11026432999999998, 0.029095632499999996, 0.0258055325, 0.025960632499999997, 0.029270532499999998]')

# Perform additional extensive tests on function compute_equilibrium
import test_correctness
ok = test_correctness.run_tests(compute_equilibrium, relies_on=[delta_x, delta_y, zero_velocity_equilibrium])

In [ ]:
# Task 3i: Test function get_upstream_cell

distribution1 = [[[0,0],[0,1]],[[1,0],[1,1]]]

print(f'get_upstream_cell(distribution1, 0, 0, Direction.REST) returns:', get_upstream_cell(distribution1, 0, 0, Direction.REST), '(expected [0, 0])')
print(f'get_upstream_cell(distribution1, 0, 0, Direction.NORTH) returns:', get_upstream_cell(distribution1, 0, 0, Direction.NORTH), '(expected None)')
print(f'get_upstream_cell(distribution1, 0, 0, Direction.SOUTH) returns:', get_upstream_cell(distribution1, 0, 0, Direction.SOUTH), '(expected  [0, 1])') 
print(f'get_upstream_cell(distribution1, 0, 0, Direction.EAST) returns:', get_upstream_cell(distribution1, 0, 0, Direction.EAST), '(expected None)') 
print(f'get_upstream_cell(distribution1, 0, 0, Direction.WEST) returns:', get_upstream_cell(distribution1, 0, 0, Direction.WEST), '(expected [1, 0])') 
print(f'get_upstream_cell(distribution1, 0, 0, Direction.NORTHEAST) returns:', get_upstream_cell(distribution1, 0, 0, Direction.NORTHEAST), '(expected None )') 
print(f'get_upstream_cell(distribution1, 0, 0, Direction.NORTHWEST) returns:', get_upstream_cell(distribution1, 0, 0, Direction.NORTHWEST), '(expected None )') 
print(f'get_upstream_cell(distribution1, 0, 0, Direction.SOUTHEAST) returns:', get_upstream_cell(distribution1, 0, 0, Direction.SOUTHEAST), '(expected None )') 
print(f'get_upstream_cell(distribution1, 0, 0, Direction.SOUTHWEST) returns:', get_upstream_cell(distribution1, 0, 0, Direction.SOUTHWEST), '(expected [1, 1])') 
print()
print(f'get_upstream_cell(distribution1, 0, 1, Direction.NORTH) returns:', get_upstream_cell(distribution1, 0, 1, Direction.NORTH), '(expected  [0, 0])')
print(f'get_upstream_cell(distribution1, 0, 1, Direction.SOUTH) returns:', get_upstream_cell(distribution1, 0, 1, Direction.SOUTH), '(expected None)') 
print(f'get_upstream_cell(distribution1, 0, 1, Direction.EAST) returns:', get_upstream_cell(distribution1, 0, 1, Direction.EAST), '(expected None)') 
print(f'get_upstream_cell(distribution1, 0, 1, Direction.WEST) returns:', get_upstream_cell(distribution1, 0, 1, Direction.WEST), '(expected [1, 1])') 
print(f'get_upstream_cell(distribution1, 0, 1, Direction.NORTHEAST) returns:', get_upstream_cell(distribution1, 0, 1, Direction.NORTHEAST), '(expected None)') 
print(f'get_upstream_cell(distribution1, 0, 1, Direction.NORTHWEST) returns:', get_upstream_cell(distribution1, 0, 1, Direction.NORTHWEST), '(expected [1, 0])') 
print(f'get_upstream_cell(distribution1, 0, 1, Direction.SOUTHEAST) returns:', get_upstream_cell(distribution1, 0, 1, Direction.SOUTHEAST), '(expected None)') 
print(f'get_upstream_cell(distribution1, 0, 1, Direction.SOUTHWEST) returns:', get_upstream_cell(distribution1, 0, 1, Direction.SOUTHWEST), '(expected None)') 
print()
print(f'get_upstream_cell(distribution1, 1, 0, Direction.NORTH) returns:', get_upstream_cell(distribution1, 1, 0, Direction.NORTH), '(expected None)')
print(f'get_upstream_cell(distribution1, 1, 0, Direction.SOUTH) returns:', get_upstream_cell(distribution1, 1, 0, Direction.SOUTH), '(expected  [1, 1])') 
print(f'get_upstream_cell(distribution1, 1, 0, Direction.EAST) returns:', get_upstream_cell(distribution1, 1, 0, Direction.EAST), '(expected [0, 0])') 
print(f'get_upstream_cell(distribution1, 1, 0, Direction.WEST) returns:', get_upstream_cell(distribution1, 1, 0, Direction.WEST), '(expected None)') 
print(f'get_upstream_cell(distribution1, 1, 0, Direction.NORTHEAST) returns:', get_upstream_cell(distribution1, 1, 0, Direction.NORTHEAST), '(expected None)') 
print(f'get_upstream_cell(distribution1, 1, 0, Direction.NORTHWEST) returns:', get_upstream_cell(distribution1, 1, 0, Direction.NORTHWEST), '(expected None)') 
print(f'get_upstream_cell(distribution1, 1, 0, Direction.SOUTHEAST) returns:', get_upstream_cell(distribution1, 1, 0, Direction.SOUTHEAST), '(expected [0, 1])') 
print(f'get_upstream_cell(distribution1, 1, 0, Direction.SOUTHWEST) returns:', get_upstream_cell(distribution1, 1, 0, Direction.SOUTHWEST), '(expected None)') 
print()
print(f'get_upstream_cell(distribution1, 1, 1, Direction.NORTH) returns:', get_upstream_cell(distribution1, 1, 1, Direction.NORTH), '(expected [1, 0])')
print(f'get_upstream_cell(distribution1, 1, 1, Direction.SOUTH) returns:', get_upstream_cell(distribution1, 1, 1, Direction.SOUTH), '(expected  None)') 
print(f'get_upstream_cell(distribution1, 1, 1, Direction.EAST) returns:', get_upstream_cell(distribution1, 1, 1, Direction.EAST), '(expected [0, 1])') 
print(f'get_upstream_cell(distribution1, 1, 1, Direction.WEST) returns:', get_upstream_cell(distribution1, 1, 1, Direction.WEST), '(expected None)') 
print(f'get_upstream_cell(distribution1, 1, 1, Direction.NORTHEAST) returns:', get_upstream_cell(distribution1, 1, 1, Direction.NORTHEAST), '(expected [0, 0])') 
print(f'get_upstream_cell(distribution1, 1, 1, Direction.NORTHWEST) returns:', get_upstream_cell(distribution1, 1, 1, Direction.NORTHWEST), '(expected None)') 
print(f'get_upstream_cell(distribution1, 1, 1, Direction.SOUTHEAST) returns:', get_upstream_cell(distribution1, 1, 1, Direction.SOUTHEAST), '(expected None)') 
print(f'get_upstream_cell(distribution1, 1, 1, Direction.SOUTHWEST) returns:', get_upstream_cell(distribution1, 1, 1, Direction.SOUTHWEST), '(expected None)') 
print()

# Perform additional extensive tests on function get_upstream_cell
import test_correctness
ok = test_correctness.run_tests(get_upstream_cell, relies_on=[delta_x, delta_y, compute_width_and_height])

<div style="background-color: #00cc00; padding:10px">

## Task 4: Implement Simulation Functions

</div> 


<p></p>

The following code cell contains definitions for functions apply_fan_force, apply_fan_and_collide_within_one_cell, apply_fan_and_collide_all_cells, stream_or_bounce_one_cell, stream_or_bounce_all_cells and simulate.

- Do not modify the name or list of parameters or header comments for any of the functions.
- Each of those functions must be implemented in the code cell below and not be copied anywhere else within this .ipynb file. Do not cut and paste to create duplicate definitions of the same function.
- Do not create any additional functions, just implement the functions already defined.
- Do not create or make use of any global variables in your functions. If you need to define a constant, make it local to your function.


In [ ]:
# Function Overview: Computes the velocity of after applying the force of the fan. If the x component of the velocity is less than the fan speed then boost it up to the fan speed.
#
# Makes use of: None
#
# Parameters:
#    velocity (tuple[float, float]): The x and y components of the current cell velocity.
#    fan_speed (float): The speed of the fan (applied horizontally).
#
# Returns:
#    tuple[float, float] the boosted velocity after applying the force of the fan.
#
def apply_fan_force(velocity, fan_speed):
    pass # delete this line and replace it by your actual implementation    
     

# Function Overview: Relaxes each cell direction value towards the cell's equilibrium. 
#                    When computing the velocity to use for the equilibrium, we boost it up to the fan speed if the current cell is a fan cell.
#
# Makes use of: compute_equilibrium, compute_velocity, compute_density, relax_towards, apply_fan_force
#
# Parameters:
#    cell (list[float]): The list of distribution values for all directions in the lattice cell.
#    relaxation_time (float): The relaxation time parameter (τ) controlling convergence towards equilibrium.
#    is_fan_cell (bool): True if the current cell is a fan cell.
#    fan_speed (float): the speed that the fan accelerates the air to (applied in the horizontal direction).
#
# Returns:
#    None (the function updates the cell in place for each direction)
#
def apply_fan_and_collide_within_one_cell(cell, relaxation_time, is_fan_cell, fan_speed):
    pass # delete this line and replace it by your actual implementation    


# Function Overview: Applies apply_fan_and_collide_within_one_cell to all non-obstacle cells in the lattice distribution.
#
# Makes use of: compute_width_and_height, apply_fan_and_collide_within_one_cell, is_fan_cell_function
#
# Parameters:
#    distribution (list[list[Optional[list[float]]]]): The 3D lattice structure where
#        distribution[x][y] is either None (obstacle) or a list of per-direction distributions for cell (x, y).
#    relaxation_time (float): The relaxation time parameter (τ) controlling convergence towards equilibrium.
#    is_fan_cell_function (callable): Predicate taking (x: int, y: int) -> bool, indicating whether a cell is in a fan region.
#    fan_speed (float): the speed that the fan accelerates the air to (applied in the horizontal direction).
#
# Returns:
#    None (the function iterates over all cells, skipping obstacles, and updates each non-obstacle cell in place)
#
def apply_fan_and_collide_all_cells(distribution, relaxation_time, is_fan_cell_function, fan_speed):
    pass # delete this line and replace it by your actual implementation    


# Function Overview: Streams distributions from upstream cells or applies bounce-back at obstacles for a single cell.
#
# Makes use of: get_upstream_cell, opposite_direction
#
# Parameters:
#    x (int): The x-coordinate of the target cell.
#    y (int): The y-coordinate of the target cell.
#    distribution_in (list[list[Optional[list[float]]]]): Input lattice where
#        distribution_in[x][y] is None for obstacle cells, otherwise a list of per-direction values.
#    distribution_out (list[list[list[float]]]): Output lattice receiving streamed or bounced values.
#        Must be shaped like `distribution_in` and pre-allocated for all non-obstacle cells.
#
# Returns:
#    None (the function updates elements within the distribution_out lattice)
#
def stream_or_bounce_one_cell(x, y, distribution_in, distribution_out):
    pass # delete this line and replace it by your actual implementation    


# Function Overview: Streams distributions or applies bounce-back across all non-obstacle cells in the lattice.
#
# Makes use of: compute_width_and_height, stream_or_bounce_one_cell
#
# Parameters:
#    distribution_in (list[list[Optional[list[float]]]]): Input lattice where
#        distribution_in[x][y] is None for obstacle cells, otherwise a list of per-direction values.
#    distribution_out (list[list[list[float]]]): Output lattice receiving streamed or bounced values.
#        Must mirror the shape of `distribution_in` and be pre-allocated for all non-obstacle cells.
#
# Returns:
#    None (the function updates elements within the distribution_out lattice)
#
def stream_or_bounce_all_cells(distribution_in, distribution_out):
    pass # delete this line and replace it by your actual implementation    
          

# Function Overview: Runs the full LBM simulation loop for a specified number of time steps, applying fan force, collide, streaming and bounce back.
#
# Makes use of: apply_fan_and_collide_all_cells, stream_or_bounce_all_cells, copy.deepcopy, report_new_distribution
#
# Parameters:
#    initial_distribution (list[list[Optional[list[float]]]]): The starting lattice distribution, including obstacles.
#    is_fan_cell_function (callable): Predicate taking (x: int, y: int) -> bool, indicating whether a cell is in a fan region.
#    fan_speed (float): Minimum enforced x-velocity for fan cells.
#    relaxation_time (float): Relaxation time parameter (τ) for relaxing toward equilibrium.
#    time_steps (int): Number of simulation steps to perform.
#    report_new_distribution (callable): Callback invoked after each time step with arguments:
#        (step: int, distribution: list) for reporting or visualization.
#
# Returns:
#    None (the function calls function report_new_distribution with the new distributions at each time step)
#
def simulate(initial_distribution, is_fan_cell_function, fan_speed, relaxation_time, time_steps, report_new_distribution):
    pass # delete this line and replace it by your actual implementation    

print('✅ All Task 4 functions defined successfully (i.e. no syntax errors). Now to see if they work correctly ...')        

In [ ]:
# Task 4a: Test function apply_fan_force

print(f'apply_fan_force((0,0), 0.05) returns:', apply_fan_force((0,0), 0.05), '(expected (0.05, 0))')   

print(f'apply_fan_force((0.025,0.021), 0.03) returns:', apply_fan_force((0.025,0.021), 0.03), '(expected (0.03, 0.021))')  

print(f'apply_fan_force((0.065,0.022), 0.05) returns:', apply_fan_force((0.065,0.022), 0.05), '(expected (0.065, 0.022))')  

# Perform additional extensive tests on function apply_fan_force
import test_correctness
ok = test_correctness.run_tests(apply_fan_force)

In [ ]:
# Task 4b: Test function apply_fan_and_collide_within_one_cell

cell3 = [0.4,0.1,0.1,0.1,0.1,0.02,0.02,0.02,0.02]

print(f'apply_fan_and_collide_within_one_cell(cell3, 0.6, False, 0.02) returns:', apply_fan_and_collide_within_one_cell(cell3, 0.6, False, 0.02), '(expected None)')      
print(f'After calling apply_fan_and_collide_within_one_cell cell3 is:\n{cell3}')
print('Expected value of cell3:\n[0.38518518518518513, 0.09629629629629628, 0.09629629629629628, 0.09629629629629628, 0.09629629629629628, 0.027407407407407405, 0.027407407407407405, 0.027407407407407405, 0.027407407407407405]')

# Perform additional extensive tests on function apply_fan_and_collide_within_one_cell
import test_correctness
ok = test_correctness.run_tests(apply_fan_and_collide_within_one_cell, relies_on=[compute_velocity, delta_x, delta_y, compute_density, apply_fan_force, compute_equilibrium, zero_velocity_equilibrium, relax_towards])

In [ ]:
# Task 4c: Test function apply_fan_and_collide_all_cells

distribution4 = [[[0.3870822432932014, 0.06562553519905999, 0.08964541121866722, 0.12087887898223304, 0.11291754584817347, 0.01674204011341538, 0.039852799611846, 0.030094629414844323, 0.009124968862628775],[0.42840434386324305, 0.10631340609634846, 0.09813423191133577, 0.11327759082483846, 0.10723144604850653, 0.027808364163573778, 0.027646298394204433, 0.025763003468988983, 0.026745530316426753]],[[0.48428830934195327, 0.1369231425354287, 0.1262382936531499, 0.11423379061040846, 0.10862372858298991, 0.02301216095332715, 0.03157298052938712, 0.037640232248597805, 0.028221973540609773],[0.4764210842071844, 0.10913236602599254, 0.12458210911287398, 0.12366873512881878, 0.12114246453504943, 0.03406114212967029, 0.02802006273614179, 0.027370864686635257, 0.03155829181024428]]]

def is_fan_cell2(x, y):
    return (x,y)==(0,2)

def is_fan_cell3(x,y): 
    return (x,y) == (0,0)

print(f'apply_fan_and_collide_all_cells(distribution4, 0.6, is_fan_cell3, 0.05) returns:', apply_fan_and_collide_all_cells(distribution4, 0.6, is_fan_cell3, 0.05), '(expected None)')  
print('After calling apply_fan_and_collide_all_cells distribution4 is:\n', distribution4)
print('Expected:\n', '[[[0.3853780365748972, 0.14314567501779774, 0.09785178989421235, 0.05786766513212295, 0.08888913700978374, 0.034629091190913217, 0.007357010233641201, 0.015254779032881277, 0.041590868457819744], [0.42640623941914607, 0.10392293154596968, 0.1091949912896777, 0.10574635518440176, 0.1099647211591286, 0.024330122050306536, 0.026023688446585924, 0.02901888403461061, 0.026716281957639216]], [[0.4850439921332498, 0.11332739047970826, 0.12136837269342965, 0.12321998771753033, 0.12606974855835107, 0.036710213635187124, 0.02967178488589832, 0.02388933988149588, 0.03145378201100142], [0.4793271841690618, 0.12410151540232243, 0.11987627483590327, 0.11919711665120353, 0.11484526794694508, 0.027419608024499657, 0.032665615687987204, 0.03124539566400401, 0.02727914199068393]]]')

# Perform additional extensive tests on function apply_fan_and_collide_all_cells
import test_correctness
ok = test_correctness.run_tests(apply_fan_and_collide_all_cells, relies_on=[compute_width_and_height, apply_fan_and_collide_within_one_cell, compute_velocity, delta_x, delta_y, compute_velocity, compute_density, apply_fan_force, compute_equilibrium, zero_velocity_equilibrium, relax_towards])

In [ ]:
# Task 4d: Test function stream_or_bounce_one_cell

import copy

input_distribution1 = [[None,None,['a0','a1','a2','a3','a4','a5','a6','a7','a8'],None,None], [None,None,['b0','b1','b2','b3','b4','b5','b6','b7','b8'],None,None], [['i0','i1','i2','i3','i4','i5','i6','i7','i8'], ['h0','h1','h2','h3','h4','h5','h6','h7','h8'], ['c0','c1','c2','c3','c4','c5','c6','c7','c8'], ['g0','g1','g2','g3','g4','g5','g6','g7','g8'], ['f0','f1','f2','f3','f4','f5','f6','f7','f8']], [None,None,['d0','d1','d2','d3','d4','d5','d6','d7','d8'],None,None],[None,None,['e0','e1','e2','e3','e4','e5','e6','e7','e8'],None,None]]
output_distribution1 = copy.deepcopy(input_distribution1)

print('stream_or_bounce_one_cell(2, 0, input_distribution1, output_distribution1) returns:', stream_or_bounce_one_cell(2, 0, input_distribution1, output_distribution1), '(expected None)')      
print('After calling stream_or_bounce_one_cell output_distribution1[2][0] is', output_distribution1[2][0])
print("Expected output_distribution1[2][0] is:'['i0', 'i3', 'i4', 'i1', 'h4', 'i7', 'i8', 'i5', 'i6']")


# Perform additional extensive tests on function stream_or_bounce_one_cell
import test_correctness
ok = test_correctness.run_tests(stream_or_bounce_one_cell, relies_on=[get_upstream_cell, delta_x, delta_y, compute_width_and_height, opposite_direction])

In [ ]:
# Task 4e: Test function stream_or_bounce_all_cells

import copy

input_distribution1 = [[None,None,['a0','a1','a2','a3','a4','a5','a6','a7','a8'],None,None], [None,None,['b0','b1','b2','b3','b4','b5','b6','b7','b8'],None,None], [['i0','i1','i2','i3','i4','i5','i6','i7','i8'], ['h0','h1','h2','h3','h4','h5','h6','h7','h8'], ['c0','c1','c2','c3','c4','c5','c6','c7','c8'], ['g0','g1','g2','g3','g4','g5','g6','g7','g8'], ['f0','f1','f2','f3','f4','f5','f6','f7','f8']], [None,None,['d0','d1','d2','d3','d4','d5','d6','d7','d8'],None,None],[None,None,['e0','e1','e2','e3','e4','e5','e6','e7','e8'],None,None]]
output_distribution1 = copy.deepcopy(input_distribution1)

print('stream_or_bounce_all_cells(input_distribution1, output_distribution1) returns:', stream_or_bounce_all_cells(input_distribution1, output_distribution1), '(expected None)')      
print('After calling stream_or_bounce_all_cells output_distribution1 is:\n', output_distribution1, "\nexpected:\n [[None, None, ['a0', 'a3', 'a4', 'b3', 'a2', 'a7', 'a8', 'a5', 'a6'], None, None], [None, None, ['b0', 'a1', 'b4', 'c3', 'b2', 'b7', 'h6', 'g7', 'b6'], None, None], [['i0', 'i3', 'i4', 'i1', 'h4', 'i7', 'i8', 'i5', 'i6'], ['h0', 'h3', 'i2', 'h1', 'c4', 'h7', 'h8', 'd7', 'b8'], ['c0', 'b1', 'h2', 'd3', 'g4', 'c7', 'c8', 'c5', 'c6'], ['g0', 'g3', 'c2', 'g1', 'f4', 'b5', 'd6', 'g5', 'g6'], ['f0', 'f3', 'g2', 'f1', 'f2', 'f7', 'f8', 'f5', 'f6']], [None, None, ['d0', 'c1', 'd4', 'e3', 'd2', 'h5', 'd8', 'd5', 'g8'], None, None], [None, None, ['e0', 'd1', 'e4', 'e1', 'e2', 'e7', 'e8', 'e5', 'e6'], None, None]]")
print()

# Perform additional extensive tests on function stream_or_bounce_all_cells
import test_correctness
ok = test_correctness.run_tests(stream_or_bounce_all_cells, relies_on=[compute_width_and_height, stream_or_bounce_one_cell, get_upstream_cell, delta_x, delta_y, opposite_direction])

In [ ]:
# Task 4f: Test function simulate

def is_fan_cell2(x, y):
    return (x,y)==(0,2)
    
def is_fan_cell3(x ,y): 
    return (x,y) == (0,0)

def output3(time, distribution):
    print('time', time)
    width, height = compute_width_and_height(distribution) 
    for y in range(height):
        for x in range(width):
            print((x,y), distribution[x][y])
    print()

distribution3 = [[[0.4444444444444444,0.1111111111111111,0.1111111111111111,0.1111111111111111,0.1111111111111111,0.027777777777777776,0.027777777777777776,0.027777777777777776,0.027777777777777776],[0.4444444444444444,0.1111111111111111,0.1111111111111111,0.1111111111111111,0.1111111111111111,0.027777777777777776,0.027777777777777776,0.027777777777777776,0.027777777777777776]],[[0.4444444444444444,0.1111111111111111,0.1111111111111111,0.1111111111111111,0.1111111111111111,0.027777777777777776,0.027777777777777776,0.027777777777777776,0.027777777777777776],[0.4444444444444444,0.1111111111111111,0.1111111111111111,0.1111111111111111,0.1111111111111111,0.027777777777777776,0.027777777777777776,0.027777777777777776,0.027777777777777776]]]

simulate(initial_distribution=distribution3, is_fan_cell_function=is_fan_cell3, fan_speed=0.05, relaxation_time=0.6, time_steps=5, report_new_distribution=output3)

# Expected Result:
"""
time 1
(0, 0) [0.4416666666666666, 0.14027777777777775, 0.11041666666666665, 0.0847222222222222, 0.11041666666666665, 0.03506944444444444, 0.02118055555555555, 0.02118055555555555, 0.03506944444444444]
(1, 0) [0.4444444444444444, 0.1111111111111111, 0.1111111111111111, 0.1111111111111111, 0.1111111111111111, 0.027777777777777776, 0.027777777777777776, 0.027777777777777776, 0.027777777777777776]
(0, 1) [0.4444444444444444, 0.1111111111111111, 0.1111111111111111, 0.1111111111111111, 0.1111111111111111, 0.027777777777777776, 0.027777777777777776, 0.027777777777777776, 0.027777777777777776]
(1, 1) [0.4444444444444444, 0.1111111111111111, 0.1111111111111111, 0.1111111111111111, 0.1111111111111111, 0.027777777777777776, 0.027777777777777776, 0.027777777777777776, 0.027777777777777776]

time 2
(0, 0) [0.41707594017528604, 0.1501917763246857, 0.10797172319939395, 0.07903051089258692, 0.10017851332285076, 0.038611070749694165, 0.0156847590212991, 0.018988192663274416, 0.03650362476203985]
(1, 0) [0.4651309541660418, 0.11373081921327532, 0.11628273854151044, 0.10076785625031237, 0.11628273854151044, 0.03329381591442994, 0.025191964062578093, 0.025191964062578093, 0.03329381591442994]
(0, 1) [0.44392950494313316, 0.11098237623578329, 0.11105993888645807, 0.11098237623578329, 0.11136858086176671, 0.027649243980873775, 0.027649243980873775, 0.027842145215441677, 0.027842145215441677]
(1, 1) [0.44972838229473877, 0.1165270077687132, 0.1165270077687132, 0.10842515591686136, 0.10842515591686136, 0.025316362014375628, 0.028108023893421173, 0.02612654719956082, 0.028108023893421173]

time 3
(0, 0) [0.39033291864748465, 0.14145481977755953, 0.10169949720548767, 0.07664522703079939, 0.09147229255116766, 0.036275867493079136, 0.011915636980768952, 0.018245667268058405, 0.03768259819457125]
(1, 0) [0.484440088011351, 0.11163774545544136, 0.12402407543747761, 0.1104923451290813, 0.12347646221859317, 0.036921575118554956, 0.025064553516550476, 0.023710663874999358, 0.03361567650720507]
(0, 1) [0.4378372636159372, 0.11237512634665829, 0.10851685370337853, 0.10829393825728903, 0.11240537032399342, 0.02728030029100703, 0.027624032920974734, 0.0274492324748874, 0.028919178487271328]
(1, 1) [0.46303704432403175, 0.12107992529984749, 0.12058017886897862, 0.10877230738635131, 0.1059168443751592, 0.024914240900923125, 0.029681658059226335, 0.02799945874932871, 0.028209335196524337]

time 4
(0, 0) [0.3870822432932014, 0.1369231425354287, 0.09813423191133577, 0.06562553519905999, 0.08964541121866722, 0.03406114212967029, 0.009124968862628775, 0.01674204011341538, 0.039852799611846]
(1, 0) [0.48428830934195327, 0.11423379061040846, 0.12458210911287398, 0.12087887898223304, 0.1262382936531499, 0.037640232248597805, 0.027646298394204433, 0.02301216095332715, 0.03157298052938712]
(0, 1) [0.42840434386324305, 0.10913236602599254, 0.10723144604850653, 0.10631340609634846, 0.11291754584817347, 0.025763003468988983, 0.026745530316426753, 0.027808364163573778, 0.028221973540609773]
(1, 1) [0.4764210842071844, 0.12366873512881878, 0.12114246453504943, 0.11327759082483846, 0.10862372858298991, 0.027370864686635257, 0.03155829181024428, 0.030094629414844323, 0.02802006273614179]

time 5
(0, 0) [0.3853780365748972, 0.14314567501779774, 0.09785178989421235, 0.05786766513212295, 0.08888913700978374, 0.034629091190913217, 0.007357010233641201, 0.015254779032881277, 0.041590868457819744]
(1, 0) [0.4850439921332498, 0.11332739047970826, 0.12136837269342965, 0.12321998771753033, 0.12606974855835107, 0.036710213635187124, 0.02967178488589832, 0.02388933988149588, 0.03145378201100142]
(0, 1) [0.42640623941914607, 0.10392293154596968, 0.1091949912896777, 0.10574635518440176, 0.1099647211591286, 0.024330122050306536, 0.026023688446585924, 0.02901888403461061, 0.026716281957639216]
(1, 1) [0.4793271841690618, 0.12410151540232243, 0.11987627483590327, 0.11919711665120353, 0.11484526794694508, 0.027419608024499657, 0.032665615687987204, 0.03124539566400401, 0.02727914199068393]
"""

# Perform additional extensive tests on function simulate
import test_correctness
ok = test_correctness.run_tests(simulate, relies_on=[stream_or_bounce_all_cells, compute_width_and_height, stream_or_bounce_one_cell, get_upstream_cell, delta_x, delta_y, opposite_direction, apply_fan_and_collide_all_cells, apply_fan_and_collide_within_one_cell, compute_velocity, compute_density, apply_fan_force, compute_equilibrium, zero_velocity_equilibrium, relax_towards])

<div style="background:#00cc00; padding:10px">
    
## Automated feedback and semi-automated marking

Running the following code will give you a good idea how well you are meeting the assessment criteria and the rough mark that you can expect to receive. Note that some of the assessment criteria are marked manually by human tutors, so you won't know your final mark until it is officially submitted and assessed.
</div>

In [ ]:
import code_analyser

# Note: relax_towards is not included in these tests as it assessed in Part A.

working = code_analyser.analyse_assessment_criteria([zero_velocity_equilibrium, delta_x, delta_y, opposite_direction, compute_density, compute_velocity, compute_speed, compute_equilibrium, apply_fan_force, apply_fan_and_collide_within_one_cell, apply_fan_and_collide_all_cells, get_upstream_cell, stream_or_bounce_one_cell, stream_or_bounce_all_cells, simulate], relies_on=[relax_towards, compute_width_and_height]) 

<div style="background:#00cc00; padding:10px">
    
## Task 5 - Experiment with Model Objects in the Wind Tunnel

</div>

<p></p>

After all the hard work, now for the ***fun part***, *playing* with our virtual wind tunnel and observing the results!

For each of the following examples, ***add markdown cells to provide some brief observations***.

Feel free to increase or decrease the figsize to suit your monitor, but keep the aspect ratio 2:1. figsize=(10,5) means 10 inches wide and 5 inches tall.

Other colour_maps that you might like to try instead of YlOrRd are: BuPu, GnBu, Oranges, OrRd, PuBu, PuRd, RdPu, Reds, YlGnBu, YlOrBr, YlOrRd.

You can try out your own model vehicle by editting the provided my_car.bmp image file.

You can experiment by changing the scale, time_steps, fan_speed and relaxation_time, but note that:
- using a larger scale will take longer to compute
- using fan speeds above 0.1 or relaxation times below 0.5 are likely to create instability in the LBM model
- using zoom = True means visualize only the central section of the wind tunnel where the model is located (but the simulation still simulates the entire space)

Note that arrows show the direction of the velocity, while each cell is colour coded based on the air speed in that cell, i.e. the magnitude of its velocity.

In [ ]:
import wind_tunnel

# Used to ensure Matplotlib figures are displayed currently within  this Jupyter environment
%matplotlib widget   

# Use the functions you have just implemented to drive an animated visualization of the wind tunnel.
animate = wind_tunnel.create_animator(compute_velocity, zero_velocity_equilibrium, simulate, working, colour_map='YlOrRd', figsize=(10, 5))

In [ ]:
# scalable_model=None is used to test the wind tunnel without placing a model object in the tunnel.

animate(scale=5, scalable_model=None, time_steps=60, zoom=False, fan_speed=0.022, relaxation_time = 0.52)

In [ ]:
# zoom in means visualize only the central section of the wind tunnel where the model object would normally be placed.

animate(scale=5, scalable_model=None, time_steps=50, zoom=True, fan_speed=0.022, relaxation_time = 0.52)

In [ ]:
# using a larger scale creates more rows and columns of cells, but takes longer to execute

animate(scale=10, scalable_model=None, time_steps=100, zoom=False, fan_speed=0.022, relaxation_time = 0.52)

In [ ]:
# let's place a solid rectangular block on the floor of our wind tunnel.

animate(scale=10, scalable_model=wind_tunnel.block1, time_steps=100, zoom=False, fan_speed=0.022, relaxation_time = 0.52)

In [ ]:
# let's zoom in on this block object

animate(scale=10, scalable_model=wind_tunnel.block1, time_steps=300, zoom=True, fan_speed=0.022, relaxation_time = 0.52)

In [ ]:
# let's elevate the block so that air can flow above and below

animate(scale=10, scalable_model=wind_tunnel.block2, time_steps=300, zoom=True, fan_speed=0.022, relaxation_time = 0.52)

In [ ]:
# does a "circular" object create a different flow pattern?

animate(scale=10, scalable_model=wind_tunnel.circle, time_steps=300, zoom=True, fan_speed=0.022, relaxation_time = 0.52)

In [ ]:
# finally, to our F1 car model, at large scale

animate(scale=20, scalable_model=wind_tunnel.image_model('images/f1.bmp'), time_steps=300, zoom=False, fan_speed=0.022, relaxation_time = 0.52)

In [ ]:
# and zoom in to see the detail

animate(scale=20, scalable_model=wind_tunnel.image_model('images/f1.bmp'), time_steps=500, zoom=True, fan_speed=0.022, relaxation_time = 0.52)

In [ ]:
# use this example to test your own model by editting the provided my_car.bmp image file

animate(scale=20, scalable_model=wind_tunnel.image_model('images/my_car.bmp'), time_steps=200, zoom=True, fan_speed=0.022, relaxation_time = 0.52)